# Practical Session 10: Deep Learning for Handwritten Digit Classification

This notebook is a fully documented model solution for the current practical script.
It uses a compact feedforward neural network on MNIST-style digit data, evaluates the model,
studies learning curves, compares model sizes, and adds regularization.


In [ ]:
# Import the standard numerical and plotting libraries.
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import scikit-learn utilities for splitting, evaluation, and the offline fallback data set.
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import load_digits

# Import TensorFlow / Keras for the neural-network models.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Silence overly verbose TensorFlow logs in the notebook output.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Fix all relevant random seeds so model training is reproducible.
SEED = 10
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# Print compact floating-point arrays.
np.set_printoptions(precision=4, suppress=True)


## Task 1: Load and inspect the data

We first try to load the Keras MNIST data set. If the environment is offline and the data
is not cached locally, we fall back automatically to the built-in `sklearn` digits data set
so that the notebook remains executable without network access.


In [ ]:
# Try to load MNIST first because it is the benchmark discussed in the script.
try:
    (X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()
    data_source = "MNIST from Keras"
except Exception:
    # Fall back to the built-in sklearn digits data set when network access is unavailable.
    digits = load_digits()
    X_all = digits.images.astype("float32")
    y_all = digits.target.astype(int)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_all,
        y_all,
        test_size=0.2,
        random_state=SEED,
        stratify=y_all,
    )
    data_source = "sklearn digits fallback"

# Report the dimensions of the original train and test splits.
print("Data source:", data_source)
print("Training images shape:", X_train_full.shape)
print("Training labels shape:", y_train_full.shape)
print("Test images shape:", X_test.shape)
print("Test labels shape:", y_test.shape)
print("Number of classes:", len(np.unique(y_train_full)))

# Show a few example images together with their labels.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
sample_indices = np.arange(10)
for ax, idx in zip(axes.ravel(), sample_indices):
    ax.imshow(X_train_full[idx], cmap="gray")
    ax.set_title(f"Label: {y_train_full[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Summarize the split sizes in a compact table.
split_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "num_samples": [len(X_train_full), len(X_test)],
    }
)
display(split_df)


## Task 2: Data preprocessing

We scale the pixel values to the interval `[0, 1]`, create a validation split, and keep a
moderate training subset so that all later experiments run quickly during the practical.


In [ ]:
# Determine the correct pixel scaling from the current data source.
pixel_scale = 255.0 if X_train_full.max() > 16 else 16.0

# Convert integer pixels to float32 and scale them to the range [0, 1].
# TODO: Scale the training images to the interval [0, 1].
X_train_full = ...
X_test = X_test.astype("float32") / pixel_scale

# Choose a validation size that works for both MNIST and the smaller offline fallback.
val_size = 2000 if len(X_train_full) > 4000 else max(250, int(0.2 * len(X_train_full)))

# Create a reproducible validation split from the original training data.
X_train_base, X_val, y_train_base, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=val_size,
    random_state=SEED,
    stratify=y_train_full,
)

# Use a moderate subset for repeated experiments so the full notebook stays fast.
desired_train_size = 12000
if len(X_train_base) > desired_train_size:
    X_train, _, y_train, _ = train_test_split(
        X_train_base,
        y_train_base,
        train_size=desired_train_size,
        random_state=SEED,
        stratify=y_train_base,
    )
else:
    X_train, y_train = X_train_base.copy(), y_train_base.copy()

# Keep labels as integer class indices because sparse cross-entropy accepts them directly.
print("Subset training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("Image shape used by the model:", X_train.shape[1:])


## Task 3: Baseline neural network

The baseline is a small fully connected classifier with two hidden layers and ReLU
activations. The output layer has ten units, one for each digit class.


In [ ]:
def build_mlp(hidden_units=(128, 64), dropout_rate=0.0, l2_strength=0.0):
    # Apply L2 regularization only when a positive strength is requested.
    kernel_regularizer = regularizers.l2(l2_strength) if l2_strength > 0 else None

    # Build the feedforward model layer by layer.
    model = keras.Sequential(name="digit_mlp")
    model.add(layers.Input(shape=X_train.shape[1:]))
    model.add(layers.Flatten(name="flatten"))
    for layer_id, units in enumerate(hidden_units, start=1):
        model.add(
            layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=kernel_regularizer,
                name=f"dense_{layer_id}",
            )
        )
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate, name=f"dropout_{layer_id}"))
    model.add(layers.Dense(10, activation="softmax", name="output"))

    # Compile the network with Adam and sparse categorical cross-entropy.
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Create and train the baseline model.
# TODO: Build the baseline MLP with two hidden layers.
baseline_model = ...
baseline_model.summary()

baseline_start = time.perf_counter()
# TODO: Train the baseline model on the training and validation data.
baseline_history = ...
baseline_training_time = time.perf_counter() - baseline_start

# Evaluate the final baseline metrics on train and validation data.
train_loss, train_acc = baseline_model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = baseline_model.evaluate(X_val, y_val, verbose=0)

architecture_df = pd.DataFrame(
    [
        {
            "hidden_layers": [128, 64],
            "trainable_parameters": baseline_model.count_params(),
            "training_time_seconds": baseline_training_time,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "validation_loss": val_loss,
            "validation_accuracy": val_acc,
        }
    ]
)
display(architecture_df)


## Task 4: Evaluation on the test set

We compute the final test accuracy, visualize the confusion matrix, and inspect correctly
and incorrectly classified examples.


In [ ]:
# Predict class probabilities on the test data and convert them to hard labels.
y_test_prob = baseline_model.predict(X_test, verbose=0)
# TODO: Convert predicted class probabilities into hard labels.
y_test_pred = ...

# Compute the final test accuracy.
test_accuracy = np.mean(y_test_pred == y_test)
print("Final test accuracy:", test_accuracy)

# Build and display the confusion matrix.
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion matrix on the test set ({data_source})")
plt.show()


def plot_examples(images, true_labels, predicted_labels, indices, title):
    # Plot a fixed number of examples with true and predicted labels.
    fig, axes = plt.subplots(1, len(indices), figsize=(12, 3))
    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx], cmap="gray")
        ax.set_title(f"t={true_labels[idx]}, p={predicted_labels[idx]}")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# Find correctly classified and misclassified test samples.
correct_idx = np.where(y_test_pred == y_test)[0][:5]
wrong_idx = np.where(y_test_pred != y_test)[0][:5]

plot_examples(X_test, y_test, y_test_pred, correct_idx, "Five correctly classified examples")
plot_examples(X_test, y_test, y_test_pred, wrong_idx, "Five misclassified examples")


## Task 5: Learning curves and overfitting

The baseline training history already contains the required learning curves for loss and
accuracy on the training and validation sets.


In [ ]:
# Extract the recorded metrics from the Keras history object.
history_df = pd.DataFrame(baseline_history.history)
history_df["epoch"] = np.arange(1, len(history_df) + 1)
display(history_df)

# Plot the loss curves and the accuracy curves side by side.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Learning curves: loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["accuracy"], marker="o", label="train")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Learning curves: accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identify the epoch with the best validation accuracy.
best_epoch = int(history_df.loc[history_df["val_accuracy"].idxmax(), "epoch"])
print("Best validation epoch:", best_epoch)


## Task 6: Effect of model size

We compare three model sizes on the same training, validation, and test splits.


In [ ]:
# Define three model sizes from small to large.
model_configs = {
    "small": (64,),
    "baseline": (128, 64),
    "large": (256, 128),
}

size_rows = []
for name, hidden_units in model_configs.items():
    # Build a fresh model for the current configuration.
    model = build_mlp(hidden_units=hidden_units)
    start = time.perf_counter()
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=4,
        batch_size=128,
        verbose=0,
    )
    duration = time.perf_counter() - start

    # Evaluate on validation and test data after training.
    _, val_acc_size = model.evaluate(X_val, y_val, verbose=0)
    _, test_acc_size = model.evaluate(X_test, y_test, verbose=0)
    size_rows.append(
        {
            "model_size": name,
            "hidden_units": list(hidden_units),
            "parameters": model.count_params(),
            "training_time_seconds": duration,
            "validation_accuracy": val_acc_size,
            "test_accuracy": test_acc_size,
        }
    )

size_df = pd.DataFrame(size_rows)
display(size_df)


## Task 7: Regularization and robustness

We add dropout and L2 regularization to the same baseline architecture and compare the
result with the unregularized model.


In [ ]:
# Build the regularized version of the baseline network.
# TODO: Build a regularized version of the baseline network.
regularized_model = ...

# Add early stopping to stop once validation performance saturates.
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True,
)

regularized_history = regularized_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=128,
    verbose=0,
    callbacks=[early_stopping],
)

# Evaluate the regularized model.
reg_train_loss, reg_train_acc = regularized_model.evaluate(X_train, y_train, verbose=0)
reg_val_loss, reg_val_acc = regularized_model.evaluate(X_val, y_val, verbose=0)

regularization_df = pd.DataFrame(
    [
        {
            "model": "baseline",
            "train_accuracy": train_acc,
            "validation_accuracy": val_acc,
            "generalization_gap": train_acc - val_acc,
        },
        {
            "model": "dropout + L2 + early stopping",
            "train_accuracy": reg_train_acc,
            "validation_accuracy": reg_val_acc,
            "generalization_gap": reg_train_acc - reg_val_acc,
        },
    ]
)
display(regularization_df)


## Task 8: Short technical reflection

Deep learning is well suited for image data because neural networks can learn complex,
nonlinear feature representations from high-dimensional pixel arrays. In this pipeline,
architectural choices such as the number of hidden layers or the use of dropout are
modeling choices, while the optimizer, learning rate, and early-stopping strategy are
optimization choices. In this experiment, the strongest practical influences were model
size, training duration, and regularization strength.
